# Akansha — MuseTalk avatar worker

Runs the photoreal talking-head model on a free Colab T4 and exposes it to your local
Akansha backend over a WebSocket tunnel. Your machine keeps the voice, the reasoning and
the browser; only face rendering happens here.

**Set the runtime to a GPU first:** Runtime → Change runtime type → **T4 GPU**. On CPU
MuseTalk renders at 1–2 fps, which is minutes of compute per sentence.

Run the cells top to bottom. The last one prints a `wss://…` URL to paste into Akansha.

### Why the worker runs *inside* this notebook

`worker.py --engine musetalk` cannot be launched as a subprocess here, and that is not a
shortcut being taken. MuseTalk's `Avatar` class reads its models (`vae`, `unet`, `pe`,
`whisper`, `audio_processor`, `timesteps`, `fp`) as **module globals** that only its
`__main__` block ever creates. `colab_bootstrap.init_runtime()` builds those globals in
*this* process; a subprocess would start with every one of them undefined and die with
`NameError` on the first render. So the notebook hosts the socket itself and hands the
already-patched engine to the same `Worker` class `worker.py` uses — identical protocol,
identical code path, one process.

In [ ]:
# 1 — confirm the GPU. Stop here if this fails; nothing below will work on CPU.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No CUDA device. Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| cuda", torch.version.cuda)

In [ ]:
# 2 — upload three files from your Akansha checkout:
#       aura/tools/musetalk_worker/worker.py
#       aura/tools/musetalk_worker/colab_bootstrap.py
#       aura/public/assets/images/akansha-presence.webp   (or your own portrait)
#
# A front-facing, well-lit head-and-shoulders crop works best. MuseTalk animates the
# mouth region of whatever you give it; it does not invent head turns or gaze.
import os, shutil
from google.colab import files

os.makedirs('/content/akansha', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'/content/akansha/{name}')
print(sorted(os.listdir('/content/akansha')))

In [ ]:
# 3 — runtime deps for the worker itself. MuseTalk's own requirements come with its
# weight-download step in the next cell.
!pip -q install websockets numpy pillow librosa
!apt-get -qq install -y ffmpeg > /dev/null && echo ffmpeg ok

In [ ]:
# 4 — clone MuseTalk, fetch weights, build the module globals, patch in the streaming
# generator, and prepare the avatar once.
#
# Preparation (face parsing + VAE latent caching) is MuseTalk's slow step, not inference.
# Doing it now rather than on the first utterance is the difference between a warm worker
# and a first sentence that arrives a minute late. Expect several minutes on the first run.
import sys
sys.path.insert(0, '/content/akansha')
import colab_bootstrap

PORTRAIT = '/content/akansha/akansha-presence.webp'   # match what you uploaded
AVATAR_ID = 'akansha'

info = colab_bootstrap.bootstrap(avatar_image=PORTRAIT, avatar_id=AVATAR_ID, version='v1')
info

Check the output of cell 4 before continuing:

- `frames` must be **greater than 0**. Zero means face detection found no face in your
  portrait, and nothing will render — try a larger, front-facing crop.
- `worker_prepared_check_passes` must be **True**. It reports whether the prepared avatar
  landed where `MuseTalkEngine._prepared()` looks (`results/avatars/<id>`, the v1 layout).
  If it is False, the engine would ask MuseTalk to re-prepare on every start and hang on an
  interactive `input()` prompt. Keep `version='v1'` above and it stays True.

In [ ]:
# 5 — start the worker on a background thread, using the same Worker class and wire
# protocol as `worker.py` run from the command line.
import asyncio, threading, logging, websockets
import worker as worker_mod

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
PORT = 8765

# Gets the *patched* Avatar out of sys.modules, and since preparation is already cached
# `_prepared()` answers True, so this constructor loads artifacts instead of rebuilding.
engine = worker_mod.MuseTalkEngine(
    musetalk_root='/content/MuseTalk',
    avatar_image=PORTRAIT,
    avatar_id=AVATAR_ID,
)
assert engine.name == 'musetalk', engine.name

_worker = worker_mod.Worker(engine)
_ready = threading.Event()

def _serve():
    async def go():
        async with websockets.serve(
            _worker.handle, '0.0.0.0', PORT, max_size=None, ping_interval=20
        ):
            _ready.set()
            await asyncio.Future()   # serve until the runtime dies
    asyncio.new_event_loop().run_until_complete(go())

threading.Thread(target=_serve, daemon=True, name='avatar-worker').start()
assert _ready.wait(timeout=120), 'worker did not start'
print(f'worker listening on ws://0.0.0.0:{PORT}  engine={engine.name}')

In [ ]:
# 6 — expose it. cloudflared needs no account and no token.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

import re, subprocess, time

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public = None
deadline = time.time() + 90
while time.time() < deadline and public is None:
    line = proc.stdout.readline()
    if not line:
        break
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if match:
        public = match.group(0)

assert public, 'cloudflared did not report a URL; re-run this cell'
# The tunnel terminates TLS, so the WebSocket scheme is wss:// even though the local
# worker is plain ws://.
wss = public.replace('https://', 'wss://')
print('\nPaste this into Akansha:\n')
print(f'  {wss}\n')
print('Or from the machine running Akansha:\n')
print(f'  curl -X POST http://127.0.0.1:8000/api/avatar/worker \\\n'
      f'    -H "Content-Type: application/json" \\\n'
      f'    -d \'{{"url":"{wss}","avatar":"{AVATAR_ID}"}}\'')

## After you paste the URL

The backend replies immediately with whether the worker is reachable:

```json
{ "worker": "connected", "model": "musetalk", "fps": 25.0, "avatars": ["akansha"] }
```

Check any time with `GET /api/avatar/status`. Nothing needs configuring in the browser —
the voice page opens `/ws/avatar/{session}` on load, is told `worker: "absent"` until a URL
is registered, and starts drawing neural frames the moment one is.

## The URL changes every restart

A Colab runtime is reclaimed after a few hours of use or ~90 minutes idle, and
`trycloudflare.com` hands out a fresh hostname each time. That is why the worker URL is a
runtime setting rather than an environment variable, and why "no worker" is a normal state
rather than an error: when this runtime dies, Akansha keeps talking mid-sentence on the CSS
rig with the same audio. Re-run cells 5 and 6 and paste the new URL.

## If the face does not move

Work outward from the model:

1. `info['frames']` from cell 4 — zero means no face was detected in your portrait.
2. This notebook's cell 5 log — a render logs frame counts per utterance. Silence there
   means the relay never forwarded a `speak`, so the problem is the URL, not the model.
3. `GET /api/avatar/status` on your machine — `worker: "absent"` means the tunnel is stale.
4. Browser devtools, `/ws/avatar/{session}` frames — a `fallback` message means the backend
   gave up on the worker and is deliberately driving the CSS rig instead.

**Untested caveat, stated plainly:** the repo this notebook belongs to has no CUDA device,
so the MuseTalk half has never been executed. The transport around it is covered by 10
passing tests against a stand-in engine. The version adaptation in `colab_bootstrap.py`
probes MuseTalk's function signatures rather than assuming them, but MuseTalk moves — if a
cell raises `TypeError` on a loader call, that is where to look.